# Emotion Modeling Notebook

This notebook builds on the preprocessing workflow and adds:
- a baseline CNN
- an improved CNN with batch normalization and dropout
- weighted cross-entropy training loops
- validation tracking
- probability outputs for confidence analysis

`CrossEntropyLoss` expects raw logits, so the models return logits. We only apply `torch.softmax(outputs, dim=1)` when we want probabilities for analysis or assistive design.

In [ ]:
import sys
!{sys.executable} -m pip install torch torchvision matplotlib

In [ ]:
import copy
import random
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from torchvision import datasets, transforms

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

In [ ]:
# Update these paths if your dataset lives somewhere else.
train_dir = Path("archive/train")
test_dir = Path("archive/test")

print("Train directory:", train_dir.resolve())
print("Test directory:", test_dir.resolve())
print("Train exists:", train_dir.exists())
print("Test exists:", test_dir.exists())

In [ ]:
train_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize((48, 48)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

eval_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize((48, 48)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

In [ ]:
class TransformSubset(Dataset):
    def __init__(self, dataset, indices, transform=None):
        self.dataset = dataset
        self.indices = list(indices)
        self.transform = transform

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        image, label = self.dataset[self.indices[idx]]
        if self.transform is not None:
            image = self.transform(image)
        return image, label

base_train_dataset = datasets.ImageFolder(train_dir)
test_dataset = datasets.ImageFolder(test_dir, transform=eval_transform)

num_classes = len(base_train_dataset.classes)
idx_to_class = {idx: name for idx, name in enumerate(base_train_dataset.classes)}

print("Classes:", base_train_dataset.classes)
print("Number of classes:", num_classes)
print("Train samples:", len(base_train_dataset))
print("Test samples:", len(test_dataset))

In [ ]:
num_train = len(base_train_dataset)
indices = torch.randperm(num_train).tolist()
train_size = int(0.8 * num_train)
train_indices = indices[:train_size]
val_indices = indices[train_size:]

train_dataset = TransformSubset(base_train_dataset, train_indices, transform=train_transform)
val_dataset = TransformSubset(base_train_dataset, val_indices, transform=eval_transform)

print("Train size:", len(train_dataset))
print("Validation size:", len(val_dataset))

In [ ]:
class_counts = Counter(base_train_dataset.targets)
total_samples = sum(class_counts.values())
class_weights = torch.tensor(
    [total_samples / class_counts[i] for i in range(num_classes)],
    dtype=torch.float32
)

train_targets = [base_train_dataset.targets[i] for i in train_indices]
sample_weights = [class_weights[label].item() for label in train_targets]
sampler = WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=True)

print("Class counts:", class_counts)
print("Class weights:", class_weights)

In [ ]:
batch_size = 64

train_loader = DataLoader(train_dataset, batch_size=batch_size, sampler=sampler)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

images, labels = next(iter(train_loader))
print("Image batch shape:", images.shape)
print("Label batch shape:", labels.shape)

In [ ]:
plt.figure(figsize=(8, 3))
plt.subplot(1, 2, 1)
plt.imshow(images[0].squeeze(), cmap="gray")
plt.title(f"Train sample: {idx_to_class[labels[0].item()]}")
plt.axis("off")

plt.subplot(1, 2, 2)
plt.bar([idx_to_class[i] for i in range(num_classes)], [class_counts[i] for i in range(num_classes)])
plt.xticks(rotation=45)
plt.title("Class distribution")
plt.tight_layout()
plt.show()

## Baseline CNN

Architecture:
- Conv -> ReLU -> Pool
- Conv -> ReLU -> Pool
- Fully connected output layer

The model returns logits. During evaluation or confidence analysis, convert logits to probabilities with `torch.softmax(outputs, dim=1)`.

In [ ]:
class BaselineCNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 12 * 12, 128),
            nn.ReLU(),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        return self.classifier(x)

baseline_model = BaselineCNN(num_classes).to(device)
baseline_model

In [ ]:
def evaluate_model(model, loader, criterion):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item() * labels.size(0)
            preds = outputs.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    return running_loss / total, correct / total


def train_model(model, train_loader, val_loader, criterion, optimizer, epochs):
    history = {
        "train_loss": [],
        "train_acc": [],
        "val_loss": [],
        "val_acc": []
    }
    best_state = None
    best_val_acc = 0.0

    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0

        for images, labels in train_loader:
            images = images.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * labels.size(0)
            preds = outputs.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

        train_loss = running_loss / total
        train_acc = correct / total
        val_loss, val_acc = evaluate_model(model, val_loader, criterion)

        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_state = copy.deepcopy(model.state_dict())

        print(
            f"Epoch {epoch + 1}/{epochs} | "
            f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | "
            f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}"
        )

    if best_state is not None:
        model.load_state_dict(best_state)

    return history


def plot_history(history, title):
    epochs = range(1, len(history["train_loss"]) + 1)
    plt.figure(figsize=(12, 4))

    plt.subplot(1, 2, 1)
    plt.plot(epochs, history["train_loss"], label="Train Loss")
    plt.plot(epochs, history["val_loss"], label="Val Loss")
    plt.title(f"{title} Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend()

    plt.subplot(1, 2, 2)
    plt.plot(epochs, history["train_acc"], label="Train Accuracy")
    plt.plot(epochs, history["val_acc"], label="Val Accuracy")
    plt.title(f"{title} Accuracy")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
baseline_criterion = nn.CrossEntropyLoss(weight=class_weights.to(device))
baseline_optimizer = torch.optim.Adam(baseline_model.parameters(), lr=1e-3)
baseline_epochs = 10

baseline_history = train_model(
    baseline_model,
    train_loader,
    val_loader,
    baseline_criterion,
    baseline_optimizer,
    baseline_epochs
)

plot_history(baseline_history, "Baseline CNN")

## Improved CNN

This version adds:
- batch normalization for more stable training
- dropout for regularization
- a slightly deeper classifier head

In [ ]:
class ImprovedCNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Dropout(0.25)
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 6 * 6, 256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        return self.classifier(x)

improved_model = ImprovedCNN(num_classes).to(device)
improved_model

In [ ]:
improved_criterion = nn.CrossEntropyLoss(weight=class_weights.to(device))
improved_optimizer = torch.optim.Adam(improved_model.parameters(), lr=1e-3, weight_decay=1e-4)
improved_epochs = 12

improved_history = train_model(
    improved_model,
    train_loader,
    val_loader,
    improved_criterion,
    improved_optimizer,
    improved_epochs
)

plot_history(improved_history, "Improved CNN")

In [ ]:
test_loss, test_acc = evaluate_model(improved_model, test_loader, improved_criterion)
print(f"Improved CNN Test Loss: {test_loss:.4f}")
print(f"Improved CNN Test Accuracy: {test_acc:.4f}")

## Output Probabilities for Confidence Analysis

Use the model's logits and convert them to probabilities with softmax.

In [ ]:
def predict_with_confidence(model, loader, idx_to_class, max_examples=5):
    model.eval()
    shown = 0

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            probs = torch.softmax(outputs, dim=1)
            confidences, preds = probs.max(dim=1)

            for i in range(images.size(0)):
                print(
                    f"True: {idx_to_class[labels[i].item()]} | "
                    f"Predicted: {idx_to_class[preds[i].item()]} | "
                    f"Confidence: {confidences[i].item():.4f}"
                )
                shown += 1
                if shown >= max_examples:
                    return


sample_images, sample_labels = next(iter(test_loader))
sample_images = sample_images.to(device)

improved_model.eval()
with torch.no_grad():
    outputs = improved_model(sample_images)
    probs = torch.softmax(outputs, dim=1)

print("Probability tensor shape:", probs.shape)
print("First sample probabilities:", probs[0])
predict_with_confidence(improved_model, test_loader, idx_to_class)

## Optional Next Step

If you want to try transfer learning later, the easiest path is to switch to a 3-channel transform and fine-tune a pretrained model such as ResNet18. For the current milestone, the baseline and improved CNNs above cover the required modeling tasks.